# PostgreSQL vs. Neo4j — Reddit Hyperlink Network Benchmark

**Course:** Data Management 2025/2026 · **Author:** Davide Timperi (1950722)

This notebook analyzes the comparative benchmark between **PostgreSQL (3NF Normalized Schema)** and **Neo4j (Multi-Node Property Graph Schema)** on the Stanford SNAP Reddit Hyperlink Network.

*(Note: Chart 1-7 have been divided into per-query subplot grids for detailed analysis)*


In [ ]:
import json, math
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

BG      = '#1e1e2e'
SURFACE = '#313244'
TEXT    = '#cdd6f4'
GRID    = '#45475a'
PG_C    = '#89b4fa'
NEO_C   = '#a6e3a1'
RED_C   = '#f38ba8'
YELLOW  = '#f9e2af'
LAVENDER= '#b4befe'

plt.rcParams.update({
    'figure.facecolor':  BG,
    'axes.facecolor':    BG,
    'axes.edgecolor':    GRID,
    'axes.labelcolor':   TEXT,
    'text.color':        TEXT,
    'xtick.color':       TEXT,
    'ytick.color':       TEXT,
    'grid.color':        GRID,
    'grid.alpha':        0.4,
    'legend.facecolor':  SURFACE,
    'legend.edgecolor':  GRID,
    'font.family':       'DejaVu Sans',
    'figure.dpi':        120,
})

RESULTS_PATH = Path('..') / 'data' / 'benchmark_results.json'


In [ ]:
with open(RESULTS_PATH) as f:
    df = pd.DataFrame(json.load(f))

for col in ['tier', 'median_execution_ms', 'cold_ms', 'stdev_execution_ms', 'cv_pct', 'buffer_hit_ratio', 'total_buffer_hits', 'total_buffer_reads', 'median_planning_ms', 'median_available_ms', 'median_consumed_ms', 'db_hits']:
    if col not in df.columns:
        df[col] = pd.NA

df['db_label']  = df['db'].map({'postgresql': 'PostgreSQL', 'neo4j': 'Neo4j'})
df['median_ms'] = df['median_execution_ms'].astype(float)
df['cold_ms_f'] = df['cold_ms'].astype(float)
df['stdev_ms']  = df['stdev_execution_ms'].fillna(0).astype(float)

QUERY_IDS = sorted(df['query_id'].unique())

def _get(df, qid, db, col):
    sub = df[(df.query_id == qid) & (df.db == db)]
    return float(sub[col].values[0]) if len(sub) and pd.notna(sub[col].values[0]) else 0


## Chart 1 — Execution Time per Query (PostgreSQL vs Neo4j)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()
for i, qid in enumerate(QUERY_IDS):
    ax = axes[i]
    pg_med = _get(df, qid, 'postgresql', 'median_ms')
    neo_med = _get(df, qid, 'neo4j', 'median_ms')
    
    bars = ax.bar(['PG', 'Neo4j'], [pg_med, neo_med], color=[PG_C, NEO_C], alpha=0.9)
    ax.set_title(qid, fontweight='bold')
    ax.set_ylabel('ms' if i % 5 == 0 else '')
    ax.yaxis.grid(True); ax.set_axisbelow(True)
    
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width()/2, h + max(pg_med, neo_med)*0.02,
                    f'{h:.1f}', ha='center', va='bottom', fontsize=9, color=TEXT)

for j in range(len(QUERY_IDS), len(axes)): axes[j].set_visible(False)
fig.suptitle('Chart 1 — Median Execution Time (Warm Runs)', fontweight='bold', y=1.02, fontsize=14)
plt.tight_layout(); plt.show()


## Chart 2 — Cold vs Warm Cache

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()
for i, qid in enumerate(QUERY_IDS):
    ax = axes[i]
    pg_cold = _get(df, qid, 'postgresql', 'cold_ms_f')
    pg_warm = _get(df, qid, 'postgresql', 'median_ms')
    neo_cold = _get(df, qid, 'neo4j', 'cold_ms_f')
    neo_warm = _get(df, qid, 'neo4j', 'median_ms')
    
    x = np.arange(2)
    w = 0.35
    ax.bar(x - w/2, [pg_cold, neo_cold], w, label='Cold', color=RED_C, alpha=0.8)
    ax.bar(x + w/2, [pg_warm, neo_warm], w, label='Warm', color=[PG_C, NEO_C], alpha=0.9)
    
    ax.set_title(qid, fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels(['PG', 'Neo4j'])
    ax.set_ylabel('ms' if i % 5 == 0 else '')
    ax.yaxis.grid(True); ax.set_axisbelow(True)
    if i == 0: ax.legend(loc='upper right', fontsize=8)

for j in range(len(QUERY_IDS), len(axes)): axes[j].set_visible(False)
fig.suptitle('Chart 2 — Cold vs Warm Cache', fontweight='bold', y=1.02, fontsize=14)
plt.tight_layout(); plt.show()


## Chart 3 — Timing Variability (Boxplots)

In [ ]:
records = []
for _, row in df.iterrows():
    times = row.get('warm_execution_ms') or []
    for t in (times if isinstance(times, list) else []):
        if isinstance(t, (int, float)) and not math.isnan(t):
            records.append({'query_id': row['query_id'], 'db': row['db_label'], 'time_ms': t})
df_runs = pd.DataFrame(records)

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()
for i, qid in enumerate(QUERY_IDS):
    ax = axes[i]
    sub = df_runs[df_runs.query_id == qid]
    pg_t = sub[sub.db == 'PostgreSQL']['time_ms'].tolist()
    neo_t = sub[sub.db == 'Neo4j']['time_ms'].tolist()
    
    bp = ax.boxplot([pg_t or [0], neo_t or [0]], tick_labels=['PG', 'Neo4j'],
                    patch_artist=True, widths=0.5,
                    medianprops={'color': RED_C, 'linewidth': 2})
    for patch, color in zip(bp['boxes'], [PG_C, NEO_C]):
        patch.set_facecolor(color); patch.set_alpha(0.8)
    ax.set_title(qid, fontweight='bold')
    ax.set_ylabel('ms' if i % 5 == 0 else '')
    ax.yaxis.grid(True); ax.set_axisbelow(True)

for j in range(len(QUERY_IDS), len(axes)): axes[j].set_visible(False)
fig.suptitle('Chart 3 — Timing Variability (5 warm runs)', fontweight='bold', y=1.02, fontsize=14)
plt.tight_layout(); plt.show()


## Chart 4 — PostgreSQL: Buffer Cache Hits vs Reads

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()
for i, qid in enumerate(QUERY_IDS):
    ax = axes[i]
    hits = _get(df, qid, 'postgresql', 'total_buffer_hits')
    reads = _get(df, qid, 'postgresql', 'total_buffer_reads')
    ratio = _get(df, qid, 'postgresql', 'buffer_hit_ratio') * 100
    
    ax.bar(['PG'], [hits], label='Cache Hits', color=PG_C, alpha=0.9)
    ax.bar(['PG'], [reads], bottom=[hits], label='Disk Reads', color=RED_C, alpha=0.8)
    
    ax.set_title(f'{qid} ({ratio:.1f}%)', fontweight='bold')
    ax.set_ylabel('Blocks' if i % 5 == 0 else '')
    ax.yaxis.grid(True); ax.set_axisbelow(True)
    if i == 0: ax.legend(loc='upper right', fontsize=8)

for j in range(len(QUERY_IDS), len(axes)): axes[j].set_visible(False)
fig.suptitle('Chart 4 — PostgreSQL Buffer Cache Hits vs Reads', fontweight='bold', y=1.02, fontsize=14)
plt.tight_layout(); plt.show()


## Chart 5 — PostgreSQL: Planning Time vs Execution Time

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()
for i, qid in enumerate(QUERY_IDS):
    ax = axes[i]
    plan_t = _get(df, qid, 'postgresql', 'median_planning_ms')
    exec_t = _get(df, qid, 'postgresql', 'median_ms')
    
    ax.bar(['PG'], [exec_t], label='Execution', color=PG_C, alpha=0.9)
    ax.bar(['PG'], [plan_t], bottom=[exec_t], label='Planning', color=LAVENDER, alpha=0.9)
    
    ax.set_title(qid, fontweight='bold')
    ax.set_ylabel('ms' if i % 5 == 0 else '')
    ax.yaxis.grid(True); ax.set_axisbelow(True)
    if i == 0: ax.legend(loc='upper right', fontsize=8)

for j in range(len(QUERY_IDS), len(axes)): axes[j].set_visible(False)
fig.suptitle('Chart 5 — PostgreSQL Planning vs Execution Time', fontweight='bold', y=1.02, fontsize=14)
plt.tight_layout(); plt.show()


## Chart 6 — Neo4j: Server Time vs Transfer Overhead

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()
for i, qid in enumerate(QUERY_IDS):
    ax = axes[i]
    avail = _get(df, qid, 'neo4j', 'median_available_ms')
    consumed = _get(df, qid, 'neo4j', 'median_consumed_ms')
    transfer = max(consumed - avail, 0)
    
    ax.bar(['Neo4j'], [avail], label='Server Exec', color=NEO_C, alpha=0.9)
    ax.bar(['Neo4j'], [transfer], bottom=[avail], label='Transfer', color=YELLOW, alpha=0.9)
    
    ax.set_title(qid, fontweight='bold')
    ax.set_ylabel('ms' if i % 5 == 0 else '')
    ax.yaxis.grid(True); ax.set_axisbelow(True)
    if i == 0: ax.legend(loc='upper right', fontsize=8)

for j in range(len(QUERY_IDS), len(axes)): axes[j].set_visible(False)
fig.suptitle('Chart 6 — Neo4j Server Time vs Transfer Overhead', fontweight='bold', y=1.02, fontsize=14)
plt.tight_layout(); plt.show()


## Chart 7 — Storage Access: PG Buffer vs Neo4j DB Hits

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()
for i, qid in enumerate(QUERY_IDS):
    ax = axes[i]
    pg_buf = _get(df, qid, 'postgresql', 'total_buffer_hits') + _get(df, qid, 'postgresql', 'total_buffer_reads')
    neo_db = _get(df, qid, 'neo4j', 'db_hits')
    
    ax.bar(['PG', 'Neo4j'], [pg_buf, neo_db], color=[PG_C, NEO_C], alpha=0.9)
    
    ax.set_title(qid, fontweight='bold')
    ax.set_ylabel('Blocks / Hits' if i % 5 == 0 else '')
    # Log scale is usually better for these huge access numbers
    ax.set_yscale('log')
    ax.yaxis.grid(True); ax.set_axisbelow(True)

for j in range(len(QUERY_IDS), len(axes)): axes[j].set_visible(False)
fig.suptitle('Chart 7 — Storage Access (PG Buffer Blocks vs Neo4j DB Hits)', fontweight='bold', y=1.02, fontsize=14)
plt.tight_layout(); plt.show()


---
## Scalability: O(1) vs O(N) Analysis

The final test examines how query performance scales as the dataset size grows (20%, 50%, 100%).
- **Neo4j** demonstrates $O(1)$ scaling for neighborhood-bounded traversals (e.g., T5-B), maintaining constant millisecond latency regardless of total graph size.
- **PostgreSQL** exhibits $O(N)$ scaling for complex joins, where performance degrades proportionally to the table size.

In [ ]:
import json
import matplotlib.pyplot as plt

try:
    with open('../data/scalability_results.json', 'r') as f:
        scal_data = json.load(f)
        
    sizes = sorted(list(set(row["dataset_pct"] for row in scal_data)))
    
    pg_times = []
    neo_times = []
    
    for size in sizes:
        pg_ms = next((row["t5b_execution_ms"] for row in scal_data if row["dataset_pct"] == size and row["db"] == "postgresql"), 0)
        neo_ms = next((row["t5b_execution_ms"] for row in scal_data if row["dataset_pct"] == size and row["db"] == "neo4j"), 0)
        pg_times.append(pg_ms)
        neo_times.append(neo_ms)
        
    fig, ax = plt.subplots(figsize=(10, 6))
    fig.suptitle('Scalability Test: Execution Time vs Dataset Size (T5-B)', fontsize=16, fontweight='bold')
    
    ax.plot(sizes, pg_times, marker='o', color=PG_C, linewidth=3, label='PostgreSQL')
    ax.plot(sizes, neo_times, marker='s', color=NEO_C, linewidth=3, label='Neo4j')
    
    ax.set_title('T5-B: Deep Neighborhood Traversal', fontweight='bold')
    ax.set_xlabel('Dataset Size (%)')
    ax.set_ylabel('Median Execution Time (ms)')
    ax.set_xticks(sizes)
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    ax.legend()
        
    plt.tight_layout()
    plt.show()

except FileNotFoundError:
    print("Scalability results not found. Please run scripts/run_scalability_test.py first.")


---
## Concurrency & Throughput Analysis

This section analyzes how both databases behave under increasing load (1, 10, 50 concurrent worker threads) by measuring Queries Per Second (QPS) and 95th Percentile (p95) Latency.
- **PostgreSQL** tradizionalmente scala molto bene sotto carico di letture concorrenti grazie al suo solido Connection Manager e modello MVCC.
- **Neo4j** sotto forte carico di mega-query analitiche in parallelo (50 threads) inizia a saturare la RAM della Java Virtual Machine, dovendo talvolta droppare le connessioni ai worker (inficiando sul QPS totale).

In [ ]:
import json
import matplotlib.pyplot as plt

try:
    with open('../data/concurrency_results.json', 'r') as f:
        conc_data = json.load(f)
        
    concurrencies = sorted(list(set(row["concurrency"] for row in conc_data)))
    
    pg_qps = []
    neo_qps = []
    pg_lat = []
    neo_lat = []
    
    for c in concurrencies:
        pg_qps.append(next((row["qps"] for row in conc_data if row["concurrency"] == c and row["db"] == "postgresql"), 0))
        neo_qps.append(next((row["qps"] for row in conc_data if row["concurrency"] == c and row["db"] == "neo4j"), 0))
        pg_lat.append(next((row["p95_latency_ms"] for row in conc_data if row["concurrency"] == c and row["db"] == "postgresql"), 0))
        neo_lat.append(next((row["p95_latency_ms"] for row in conc_data if row["concurrency"] == c and row["db"] == "neo4j"), 0))
        
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    fig.suptitle('Concurrency & Throughput Test (1, 10, 50 Workers)', fontsize=16, fontweight='bold')
    
    # QPS Plot
    ax1.plot(concurrencies, pg_qps, marker='o', color=PG_C, linewidth=3, label='PostgreSQL')
    ax1.plot(concurrencies, neo_qps, marker='s', color=NEO_C, linewidth=3, label='Neo4j')
    ax1.set_title('Throughput: Queries Per Second (QPS)', fontweight='bold')
    ax1.set_xlabel('Concurrent Workers')
    ax1.set_ylabel('QPS (Higher is Better)')
    ax1.set_xticks(concurrencies)
    ax1.grid(axis='y', linestyle='--', alpha=0.7)
    ax1.legend()
    
    # P95 Latency Plot
    ax2.plot(concurrencies, pg_lat, marker='o', color=PG_C, linewidth=3, label='PostgreSQL')
    ax2.plot(concurrencies, neo_lat, marker='s', color=NEO_C, linewidth=3, label='Neo4j')
    ax2.set_title('p95 Latency', fontweight='bold')
    ax2.set_xlabel('Concurrent Workers')
    ax2.set_ylabel('Latency (ms) (Lower is Better)')
    ax2.set_xticks(concurrencies)
    ax2.grid(axis='y', linestyle='--', alpha=0.7)
    ax2.legend()
        
    plt.tight_layout()
    plt.show()

except FileNotFoundError:
    print("Concurrency results not found. Please run scripts/run_concurrency.py first.")
